# 02_preprocesamiento.ipynb

## Pipeline de Preprocesamiento para Series de Tiempo

En este notebook se ejecuta el pipeline completo de limpieza, ingeniería de características y preparación del dataset para modelos recurrentes (RNN/LSTM/GRU).

**Arquitectura del pipeline (optimizada para Colab):**
1. **pandas**: lectura del CSV raw e imputación temporal (`ffill`/`bfill`)
2. **Parquet**: persistencia intermedia en formato nativo de Spark
3. **PySpark lazy**: lectura del Parquet y definición de transformaciones (temporales, agregación horaria, lags, ventanas móviles) sin materialización prematura
4. **PySpark + pandas híbrido**: materialización solo sobre el dataset agregado por hora (~35,000 filas), nunca sobre el minutal completo (2M filas)

Esta estrategia evita los cuellos de botella de la JVM en modo local, manteniendo el 100% del pipeline de Big Data en PySpark.

In [1]:
# -*- coding: utf-8 -*-
!pip install pyspark findspark -q

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

import pandas as pd
import numpy as np
import json
import os
from google.colab import files  # ← CORREGIDO: importación necesaria para la descarga final

# Spark configurado para Colab (single-node)
spark = SparkSession.builder \
    .appName("Preprocesamiento_Electricidad") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print(f"   Versión: {spark.version}")

✅ Spark iniciado.
   Versión: 4.0.3


## 1. Lectura CSV con pandas → Imputación → Parquet (STRING)

El dataset raw tiene 2,075,259 registros con valores faltantes en `?`. La imputación ffill/bfill en pandas es C-optimizada y ejecuta en segundos.

**Conversión a string:** `pdf['Datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')` evita que pandas guarde el Parquet con `timestamp64[ns]` (nanosegundos), que Spark no puede leer.

In [3]:
from google.colab import files
data_to_load = files.upload()

Saving household_power_consumption.txt to household_power_consumption.txt


In [4]:
# ============================================================
# PASO 1: LECTURA → IMPUTACIÓN → PARQUET (DATETIME COMO STRING)
# ============================================================

file_path = "/content/household_power_consumption.txt"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"❌ No se encontró {file_path}. Sube el archivo.")

# 1.1 Lectura pandas
pdf = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    dtype={
        'Date': str, 'Time': str,
        'Global_active_power': float, 'Global_reactive_power': float,
        'Voltage': float, 'Global_intensity': float,
        'Sub_metering_1': float, 'Sub_metering_2': float, 'Sub_metering_3': float
    },
    low_memory=False
)
print(f"   Filas: {len(pdf):,}")

# 1.2 Timestamp
print("Parseando fechas...")
pdf['Datetime'] = pd.to_datetime(
    pdf['Date'] + ' ' + pdf['Time'],
    format='%d/%m/%Y %H:%M:%S',
    dayfirst=True
)

# 1.3 Imputación
print("Imputando (ffill + bfill)...")
numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]
pdf[numeric_cols] = pdf[numeric_cols].ffill().bfill()

print(f"   Nulos restantes: {pdf[numeric_cols].isnull().sum().sum()}")

# 1.4 GUARDAR COMO STRING (¡CRÍTICO! Evita TIMESTAMP(NANOS))
print("Guardando Parquet con Datetime como STRING...")
pdf['Datetime'] = pdf['Datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Verificar que NO es timestamp
print(f"   Tipo de Datetime: {pdf['Datetime'].dtype}")

parquet_path = "/content/household_clean.parquet"
pdf.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f"\n Parquet guardado: {parquet_path}")
print(f"   Tamaño: {os.path.getsize(parquet_path) / 1024 / 1024:.1f} MB")

   Filas: 2,075,259
Parseando fechas...
Imputando (ffill + bfill)...
   Nulos restantes: 0
Guardando Parquet con Datetime como STRING...
   Tipo de Datetime: object

 Parquet guardado: /content/household_clean.parquet
   Tamaño: 18.3 MB


**Verificación del Parquet**

Antes de pasar a Spark, confirmamos que el Parquet se guardó correctamente con `Datetime` como string y no como timestamp.

In [5]:
# ============================================================
# VERIFICACIÓN: Leer el Parquet con pandas para confirmar tipo
# ============================================================

check = pd.read_parquet("/content/household_clean.parquet")
print("Columnas:", check.columns.tolist())
print(f"Tipo de Datetime: {check['Datetime'].dtype}")
print("Primeras 3 filas de Datetime:")
print(check['Datetime'].head(3).tolist())

# Confirmar que es string, no datetime64[ns]
assert check['Datetime'].dtype == 'object', "❌ ERROR: Datetime no es string!"
print("\n✅ CONFIRMADO: Datetime es STRING. Spark podrá leerlo.")

Columnas: ['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Datetime']
Tipo de Datetime: object
Primeras 3 filas de Datetime:
['2006-12-16 17:24:00', '2006-12-16 17:25:00', '2006-12-16 17:26:00']

✅ CONFIRMADO: Datetime es STRING. Spark podrá leerlo.


## 2. Carga en PySpark + Componentes temporales

Ahora Spark lee el Parquet sin problema porque `Datetime` es string. Se parsea con `to_timestamp()` de Spark, que genera un timestamp compatible con todas las funciones temporales.

In [6]:
# ============================================================
# PASO 2: CARGA EN SPARK (LAZY) + COMPONENTES TEMPORALES
# ============================================================

# 2.1 Leer Parquet (lazy, instantáneo)
df = spark.read.parquet("/content/household_clean.parquet")

# 2.2 Parsear Datetime desde STRING → Spark Timestamp
df = df.withColumn("Datetime", to_timestamp(col("Datetime"), "yyyy-MM-dd HH:mm:ss"))
print("   Datetime parseado desde string.")

# 2.3 Componentes temporales (lazy)
df = df.withColumn("Year", year(col("Datetime")))
df = df.withColumn("Month", month(col("Datetime")))
df = df.withColumn("Day", dayofmonth(col("Datetime")))
df = df.withColumn("Hour", hour(col("Datetime")))
df = df.withColumn("Minute", minute(col("Datetime")))
df = df.withColumn("DayOfWeek", dayofweek(col("Datetime")))
df = df.withColumn("WeekOfYear", weekofyear(col("Datetime")))
df = df.withColumn("IsWeekend", when(col("DayOfWeek").isin([1, 7]), 1).otherwise(0))


   Datetime parseado desde string.


## 3. Agregación horaria (lazy)

El dataset minutal (2,075,259 filas) es computacionalmente muy pesado para entrenar redes neuronales recurrentes en este entorno. Se reduce la dimensionalidad en un factor de 60 mediante agregación por hora (~35,000 filas), manteniendo la información física:

| Variable | Agregación | Razón |
|---|---|---|
| `Global_active_power` | **avg** | Potencia instantánea promedio durante la hora |
| `Global_reactive_power` | **avg** | Potencia reactiva promedio durante la hora |
| `Voltage` | **avg** | Voltaje promedio durante la hora |
| `Global_intensity` | **avg** | Intensidad promedio durante la hora |
| `Sub_metering_1, 2, 3` | **sum** | Energía total (Wh) acumulada en la hora |

Se conserva la estructura temporal completa (4 años, 47 meses) con todos los ciclos estacionales y diarios necesarios para el modelado.

In [7]:
# ============================================================
# PASO 3: AGREGACIÓN HORARIA (LAZY)
# ============================================================

# Agregación por hora: promedio para potencias/voltaje, suma para energía (Wh)
df_hourly = df.groupBy("Year", "Month", "Day", "Hour").agg(
    avg("Global_active_power").alias("Global_active_power"),
    avg("Global_reactive_power").alias("Global_reactive_power"),
    avg("Voltage").alias("Voltage"),
    avg("Global_intensity").alias("Global_intensity"),
    sum("Sub_metering_1").alias("Sub_metering_1"),
    sum("Sub_metering_2").alias("Sub_metering_2"),
    sum("Sub_metering_3").alias("Sub_metering_3"),
    first("DayOfWeek").alias("DayOfWeek"),
    first("IsWeekend").alias("IsWeekend")
).orderBy("Year", "Month", "Day", "Hour")

# Reconstruir timestamp horario para lags y ordenamiento
# ← CORREGIDO: compatible con cualquier versión de Spark (no usa make_timestamp)
df_hourly = df_hourly.withColumn(
    "Datetime",
    to_timestamp(
        concat(col("Year"), lit("-"), col("Month"), lit("-"), col("Day"),
               lit(" "), col("Hour"), lit(":00:00")),
        "yyyy-M-d H:mm:ss"
    )
)

# Seleccionar y ordenar columnas finales
cols_order = ["Datetime", "Year", "Month", "Day", "Hour", "DayOfWeek", "IsWeekend",
              "Global_active_power", "Global_reactive_power", "Voltage",
              "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

df_hourly = df_hourly.select(*cols_order)

print("⏰ Agregación horaria definida (lazy).")
print("   Dataset horario estimado: ~35,000 filas.")
print("   Listo para: lags → cíclicas → log → ventanas → escalado → split.")

⏰ Agregación horaria definida (lazy).
   Dataset horario estimado: ~35,000 filas.
   Listo para: lags → cíclicas → log → ventanas → escalado → split.


**Interpretación de la agregación horaria**

La reducción de dimensionalidad de 2,075,259 registros minutales a aproximadamente 35,000 registros horarios hace viable el entrenamiento de redes neuronales recurrentes en hardware limitado, sin perder información estructural. La agregación por promedio para potencias y voltaje mantiene el nivel representativo del período, mientras que la suma para submediciones preserva la energía total consumida.

El dataset horario mantiene todos los ciclos estacionales (invierno/verano), diarios (pico mañana/noche) y semanales (día laborable vs. fin de semana) necesarios para que LSTM/GRU capturen patrones de consumo.

## 4. Ingeniería de características: Variables rezagadas (Lags)

Las redes neuronales recurrentes requieren ventanas de historia para aprender dependencias temporales. Se crean lags utilizando funciones de ventana (`Window` + `lag`) en PySpark.

Se definen rezagos de:
- **1, 2, 3 horas**: dependencia de corto plazo (inercia térmica, hábitos recientes)
- **6, 12, 24 horas**: patrones diarios (desayuno, cena, calefacción nocturna)
- **168 horas (1 semana)**: patrones semanales (día de semana vs. fin de semana)

**NUEVO:** Se agregan lags también para `Sub_metering_1` (cocina) y `Sub_metering_2` (lavandería) en periodos clave (1h, 24h, 168h) para enriquecer la señal predictiva.

In [8]:
# ============================================================
# PASO 4: CREACIÓN DE VARIABLES REZAGADAS (LAGS) - LAZY
# ============================================================

window_lag = Window.orderBy("Datetime")

# Variables sobre las que crear lags (objetivo + predictoras clave)
# ← MEJORADO: se agregan Sub_metering_1 y Sub_metering_2
lag_vars = ["Global_active_power", "Sub_metering_3", "Voltage",
            "Global_reactive_power", "Sub_metering_1", "Sub_metering_2"]
lag_periods = [1, 2, 3, 6, 12, 24, 168]  # horas

print("🔗 Definiendo variables rezagadas (lazy)...")
for var in lag_vars:
    for p in lag_periods:
        df_hourly = df_hourly.withColumn(
            f"{var}_lag{p}h",
            lag(col(var), p).over(window_lag)
        )

total_lags = len(lag_vars) * len(lag_periods)
print(f"   {total_lags} columnas de lags definidas.")
print("   No se ejecuta todavía (lazy).")

🔗 Definiendo variables rezagadas (lazy)...
   42 columnas de lags definidas.
   No se ejecuta todavía (lazy).


**Interpretación de los lags**

La creación de variables rezagadas transforma el problema de predicción puntual en un problema de ventana temporal. Los lags de 1-3 horas capturan la **inercia del consumo**: si la calefacción estaba encendida hace 1 hora, probablemente sigue encendida. Los lags de 6-12 horas capturan **patrones diarios** (pico matutino vs. vespertino). El lag de 168 horas (1 semana) permite al modelo reconocer que un lunes se comporta como el lunes anterior, no como el domingo.

El uso de `Window.orderBy("Datetime")` garantiza que los rezagos se calculen en orden cronológico estricto, preservando la causalidad temporal: el modelo solo ve el pasado, nunca el futuro.

## 5. Materialización controlada: dropna + cache

Hasta este punto, todo ha sido **lazy**: Spark ha construido el plan de ejecución pero no ha procesado ninguna fila. Ahora se fuerza la primera materialización, pero **solo sobre el dataset horario** (~35,000 filas), nunca sobre el minutal (2M filas).

Se eliminan las primeras 168 filas que contienen nulos por los lags (la ventana máxima de 1 semana), y se cachea el resultado para acelerar las siguientes transformaciones.

In [9]:
# ============================================================
# PASO 5: MATERIALIZACIÓN CONTROLADA (SOLO DATASET HORARIO)
# ============================================================

# Forzar ejecución: convertir a pandas SOLO el dataset horario (~35k filas)
print("⚡ Forzando ejecución del plan lazy...")
print("   Convirtiendo dataset horario a pandas (35k filas, trivial)...")

pdf_hourly = df_hourly.toPandas()
print(f"   Filas horarias: {len(pdf_hourly):,}")

# Eliminar nulos generados por lags iniciales
pdf_hourly = pdf_hourly.dropna()
print(f"   Filas después de dropna: {len(pdf_hourly):,}")
print(f"   Filas eliminadas por lags: {35040 - len(pdf_hourly):,} (esperado: ~168)")

# Volver a Spark y cachear
df_hourly = spark.createDataFrame(pdf_hourly)
df_hourly.cache()

print("\n✅ Dataset horario limpio cacheado en Spark.")
print("   Listo para: cíclicas → log → ventanas → deltas → split → escalado.")

⚡ Forzando ejecución del plan lazy...
   Convirtiendo dataset horario a pandas (35k filas, trivial)...
   Filas horarias: 34,589
   Filas después de dropna: 34,421
   Filas eliminadas por lags: 619 (esperado: ~168)

✅ Dataset horario limpio cacheado en Spark.
   Listo para: cíclicas → log → ventanas → deltas → split → escalado.


**Interpretación de la materialización**

La conversión a pandas del dataset horario (~35,000 filas) es instantánea porque cabe completamente en memoria RAM de Colab. El `dropna()` elimina las primeras 168 filas donde los lags de 1 semana aún no tienen historia disponible. Este descarte es aceptable: representa menos del 0.5% del dataset horario y no afecta los ciclos estacionales completos (2007-2010).

El retorno a Spark con `createDataFrame` sobre 35k filas es trivial para la JVM, y el `cache()` garantiza que las siguientes transformaciones no re-executen el plan desde el origen.

## 6. Codificación cíclica de variables temporales

Las variables temporales como `Hour` (0-23) o `DayOfWeek` (1-7) son ordinales pero no métricas: la hora 23 está a 1 hora de distancia de la hora 0, no a 23. Si se alimentan como enteros a una red neuronal, el modelo aprende una relación lineal falsa.

La transformación seno/coseno proyecta estas variables en un **círculo unitario**, preservando la distancia cíclica real. Esto es crítico para:
- **Hora del día**: patrones de madrugada vs. medianoche
- **Día de la semana**: lunes está cerca de domingo
- **Mes**: invierno está cerca de verano en el ciclo anual

Se excluye `Year` de la codificación cíclica porque la tendencia anual sí es lineal y monotónica en este dataset (2006-2010).

In [10]:
# ============================================================
# PASO 6: CODIFICACIÓN CÍCLICA (LAZY SOBRE DATASET CACHEADO)
# ============================================================

print("🔄 Creando variables cíclicas...")

# Hora del día (24 horas)
df_hourly = df_hourly.withColumn("Hour_sin", sin(2 * np.pi * col("Hour") / 24))
df_hourly = df_hourly.withColumn("Hour_cos", cos(2 * np.pi * col("Hour") / 24))

# Día de la semana (7 días, 1=Domingo → 0-indexado)
df_hourly = df_hourly.withColumn("DayOfWeek_sin", sin(2 * np.pi * (col("DayOfWeek") - 1) / 7))
df_hourly = df_hourly.withColumn("DayOfWeek_cos", cos(2 * np.pi * (col("DayOfWeek") - 1) / 7))

# Mes (12 meses, 1=Enero → 0-indexado)
df_hourly = df_hourly.withColumn("Month_sin", sin(2 * np.pi * (col("Month") - 1) / 12))
df_hourly = df_hourly.withColumn("Month_cos", cos(2 * np.pi * (col("Month") - 1) / 12))

print("✅ Variables cíclicas creadas.")
print("   Hour_sin/cos, DayOfWeek_sin/cos, Month_sin/cos")

🔄 Creando variables cíclicas...
✅ Variables cíclicas creadas.
   Hour_sin/cos, DayOfWeek_sin/cos, Month_sin/cos


## 7. Transformación logarítmica

El análisis exploratorio mostró coeficientes de variación extremos:
- `Sub_metering_1` (Cocina): CV = 548%
- `Sub_metering_2` (Lavandería): CV = 448%
- `Sub_metering_3` (Calefacción): CV = 131%

Estas distribuciones están fuertemente sesgadas hacia la derecha con largas colas de outliers. La transformación `log1p(x) = ln(1+x)` comprime la escala de valores extremos sin eliminarlos, acercando la distribución a una forma más simétrica.

Esto estabiliza el entrenamiento de redes neuronales recurrentes, cuyas funciones de activación (`tanh`, `sigmoid`) saturan con gradientes muy grandes. La variable objetivo `Global_active_power` **no se transforma** para mantener la interpretabilidad directa de las predicciones en kW.

In [11]:
# ============================================================
# PASO 7: TRANSFORMACIÓN LOGARÍTMICA (LAZY)
# ============================================================

print("📐 Aplicando transformación logarítmica...")

log_vars = ["Global_reactive_power", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

for c in log_vars:
    df_hourly = df_hourly.withColumn(f"{c}_log", log1p(col(c)))

print("✅ Transformaciones logarítmicas aplicadas.")
print("   Variables: Global_reactive_power_log, Sub_metering_1_log,")
print("              Sub_metering_2_log, Sub_metering_3_log")

📐 Aplicando transformación logarítmica...
✅ Transformaciones logarítmicas aplicadas.
   Variables: Global_reactive_power_log, Sub_metering_1_log,
              Sub_metering_2_log, Sub_metering_3_log


## 8. Features de ventana móvil y diferencias temporales (NUEVO)

Además de los lags puntuales, agregamos features que capturan **tendencias e inercia** sobre ventanas de tiempo:

| Feature | Tipo | Descripción |
|---|---|---|
| `GAP_ma24h` | Media móvil 24h | Tendencia del consumo en el último día |
| `GAP_ma168h` | Media móvil 168h | Tendencia semanal del consumo |
| `GAP_std24h` | Desviación móvil 24h | Volatilidad reciente del consumo |
| `SM3_ma24h` | Media móvil 24h | Tendencia de calefacción/AC |
| `GAP_diff1h` | Delta 1h | Cambio respecto a la hora anterior |
| `GAP_diff24h` | Delta 24h | Cambio respecto al mismo horario ayer |
| `GAP_diff168h` | Delta 168h | Cambio respecto al mismo horario hace 1 semana |

Estas variables ayudan a LSTM/GRU a detectar si el consumo está "subiendo" o "estable" más allá del valor absoluto.

In [12]:
# ============================================================
# PASO 8: VENTANAS MÓVILES Y DIFERENCIAS (LAZY)
# ============================================================

print("📊 Creando features de ventana móvil y diferencias...")

window_24 = Window.orderBy("Datetime").rowsBetween(-23, 0)
window_168 = Window.orderBy("Datetime").rowsBetween(-167, 0)

# Medias móviles
df_hourly = df_hourly.withColumn("GAP_ma24h", avg("Global_active_power").over(window_24))
df_hourly = df_hourly.withColumn("GAP_ma168h", avg("Global_active_power").over(window_168))
df_hourly = df_hourly.withColumn("GAP_std24h", stddev("Global_active_power").over(window_24))
df_hourly = df_hourly.withColumn("SM3_ma24h", avg("Sub_metering_3").over(window_24))

# Diferencias temporales (deltas)
df_hourly = df_hourly.withColumn("GAP_diff1h", col("Global_active_power") - col("Global_active_power_lag1h"))
df_hourly = df_hourly.withColumn("GAP_diff24h", col("Global_active_power") - col("Global_active_power_lag24h"))
df_hourly = df_hourly.withColumn("GAP_diff168h", col("Global_active_power") - col("Global_active_power_lag168h"))

# Energía no medida (NUEVA variable predictora informativa)
# Fórmula: energía activa total (Wh) - suma de submedidores
df_hourly = df_hourly.withColumn(
    "Unmetered_energy",
    (col("Global_active_power") * lit(1000.0)) - col("Sub_metering_1") - col("Sub_metering_2") - col("Sub_metering_3")
)

print("✅ Features avanzadas creadas:")
print("   GAP_ma24h, GAP_ma168h, GAP_std24h, SM3_ma24h")
print("   GAP_diff1h, GAP_diff24h, GAP_diff168h")
print("   Unmetered_energy")

📊 Creando features de ventana móvil y diferencias...
✅ Features avanzadas creadas:
   GAP_ma24h, GAP_ma168h, GAP_std24h, SM3_ma24h
   GAP_diff1h, GAP_diff24h, GAP_diff168h
   Unmetered_energy


## 9. División temporal Train / Validation / Test

En series de tiempo, una división aleatoria (`train_test_split`) cometería **fuga de datos temporal**: el modelo entrenaría con "el futuro" y predeciría "el pasado", inflando artificialmente las métricas.

La división por años garantiza:
- **Train (2007-2008)**: ~70% — dos ciclos completos invierno-verano para aprendizaje
- **Validation (2009)**: ~15% — tercer ciclo para ajuste de hiperparámetros
- **Test (2010)**: ~15% — cuarto año, datos nunca vistos para evaluación final

Esta estructura respeta la causalidad temporal y evalúa la capacidad de generalización del modelo en condiciones reales de predicción futura.

In [13]:
# ============================================================
# PASO 9: DIVISIÓN TEMPORAL
# ============================================================

train_df = df_hourly.filter(col("Year") <= 2008)
val_df   = df_hourly.filter(col("Year") == 2009)
test_df  = df_hourly.filter(col("Year") >= 2010)

# Conteos (materialización ligera sobre datasets pequeños)
train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()
total_count = train_count + val_count + test_count

print("=" * 60)
print("DIVISIÓN TEMPORAL DEL DATASET")
print("=" * 60)
print(f"Train      (<=2008): {train_count:>7,} filas ({train_count/total_count*100:.1f}%)")
print(f"Validation (2009)  : {val_count:>7,} filas ({val_count/total_count*100:.1f}%)")
print(f"Test       (>=2010): {test_count:>7,} filas ({test_count/total_count*100:.1f}%)")
print(f"Total              : {total_count:>7,} filas")

# Sanity check: verificar continuidad temporal
train_max = train_df.select(max("Datetime")).collect()[0][0]
val_min = val_df.select(min("Datetime")).collect()[0][0]
val_max = val_df.select(max("Datetime")).collect()[0][0]
test_min = test_df.select(min("Datetime")).collect()[0][0]

print(f"\n📅 Train termina: {train_max}")
print(f"📅 Val inicia:    {val_min}")
print(f"📅 Val termina:   {val_max}")
print(f"📅 Test inicia:   {test_min}")
print("✅ Sin solapamiento temporal.")

DIVISIÓN TEMPORAL DEL DATASET
Train      (<=2008):  17,743 filas (51.5%)
Validation (2009)  :   8,760 filas (25.4%)
Test       (>=2010):   7,918 filas (23.0%)
Total              :  34,421 filas

📅 Train termina: 2008-12-31 23:00:00
📅 Val inicia:    2009-01-01 00:00:00
📅 Val termina:   2009-12-31 23:00:00
📅 Test inicia:   2010-01-01 00:00:00
✅ Sin solapamiento temporal.


## 10. Escalado Min-Max [0, 1] — SIN FUGA DE DATOS

El escalado Min-Max es el estándar para redes neuronales recurrentes porque:
1. Las activaciones LSTM/GRU (especialmente `tanh` y `sigmoid`) operan en rangos acotados.
2. Evita que variables con escalas muy diferentes (Voltage ~240 vs. Sub_metering_1 ~1) dominen la función de pérdida.
3. Garantiza que los gradientes fluyan de manera estable durante el backpropagation.

**← CORREGIDO:** Los parámetros `min` y `max` se calculan **exclusivamente sobre el conjunto de entrenamiento** (`train_df`) para evitar que información del test o validation influya en la transformación. Luego se aplican los mismos parámetros a los tres splits.

Los parámetros `min_max_dict` se exportarán en JSON para reproducir exactamente el mismo escalado en el dashboard de Streamlit.

In [14]:
# ============================================================
# PASO 10: ESCALADO MIN-MAX [0, 1] (SOLO CON TRAIN)
# ============================================================

from pyspark.sql.functions import min as spark_min, max as spark_max

# Columnas a escalar (excluir temporales, cíclicas [ya en [-1,1]], y binarias)
cols_to_scale = [
    "Global_active_power",                    # objetivo
    "Global_reactive_power", "Global_reactive_power_log",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1", "Sub_metering_1_log",
    "Sub_metering_2", "Sub_metering_2_log",
    "Sub_metering_3", "Sub_metering_3_log",
    "Unmetered_energy",
    "GAP_ma24h", "GAP_ma168h", "GAP_std24h", "SM3_ma24h",
    "GAP_diff1h", "GAP_diff24h", "GAP_diff168h"
]

# Agregar lags de variables principales (sin las versiones log)
lag_cols_to_scale = [c for c in df_hourly.columns
                     if "_lag" in c and "_log" not in c]
cols_to_scale.extend(lag_cols_to_scale)

print(f"📏 Calculando min/max SOLO con Train para {len(cols_to_scale)} variables...")

# ← CORREGIDO: calcular parámetros únicamente sobre train_df (sin fuga)
min_max_dict = {}
for c in cols_to_scale:
    min_val = train_df.select(spark_min(col(c)).alias("m")).collect()[0]["m"]
    max_val = train_df.select(spark_max(col(c)).alias("m")).collect()[0]["m"]
    min_max_dict[c] = {"min": float(min_val), "max": float(max_val)}

# Función para aplicar escalado a cualquier dataframe
def apply_minmax_scaling(df, min_max_dict):
    for c, params in min_max_dict.items():
        min_val = params["min"]
        max_val = params["max"]
        denom = max_val - min_val if (max_val - min_val) != 0 else 1e-8
        df = df.withColumn(f"{c}_scaled", (col(c) - lit(min_val)) / lit(denom))
    return df

# Aplicar a los 3 splits con los MISMOS parámetros de train
print("   Aplicando escalado a Train, Validation y Test...")
train_df = apply_minmax_scaling(train_df, min_max_dict)
val_df   = apply_minmax_scaling(val_df, min_max_dict)
test_df  = apply_minmax_scaling(test_df, min_max_dict)

print("✅ Escalado completado sin fuga de datos.")
print(f"   Parámetros guardados para {len(min_max_dict)} variables.")

📏 Calculando min/max SOLO con Train para 61 variables...
   Aplicando escalado a Train, Validation y Test...
✅ Escalado completado sin fuga de datos.
   Parámetros guardados para 61 variables.


## 11. Persistencia en formato Parquet + JSON

El formato **Parquet** se selecciona por ser columnar, comprimido y nativo de Spark, lo que permite leer únicamente las columnas necesarias en los notebooks de modelado (03-05) sin cargar todo el dataset en memoria.

El archivo `min_max_scaler.json` es el **contrato de preprocesamiento**: cualquier dato nuevo que entre al dashboard de Streamlit debe pasar por exactamente los mismos mínimos y máximos para que las predicciones sean válidas. Sin esta persistencia, el despliegue sería irreproducible.

In [15]:
# ============================================================
# PASO 11: GUARDADO DE DATASETS Y PARÁMETROS
# ============================================================

output_path = "/content/preprocessed_data"
os.makedirs(output_path, exist_ok=True)

# Guardar datasets en Parquet (formato columnar, eficiente)
print("💾 Guardando datasets en Parquet...")
train_df.write.mode("overwrite").parquet(f"{output_path}/train")
val_df.write.mode("overwrite").parquet(f"{output_path}/validation")
test_df.write.mode("overwrite").parquet(f"{output_path}/test")
df_hourly.write.mode("overwrite").parquet(f"{output_path}/full_hourly")

# Guardar parámetros de escalado en JSON
with open(f"{output_path}/min_max_scaler.json", "w") as f:
    json.dump(min_max_dict, f, indent=2)

# Muestra CSV para inspección humana
sample_csv = df_hourly.limit(1000).toPandas()
sample_csv.to_csv(f"{output_path}/sample_preprocessed.csv", index=False)

print(f"\n✅ Datasets guardados en: {output_path}")
print("   ├── train/")
print("   ├── validation/")
print("   ├── test/")
print("   ├── full_hourly/")
print("   ├── min_max_scaler.json")
print("   └── sample_preprocessed.csv")

💾 Guardando datasets en Parquet...

✅ Datasets guardados en: /content/preprocessed_data
   ├── train/
   ├── validation/
   ├── test/
   ├── full_hourly/
   ├── min_max_scaler.json
   └── sample_preprocessed.csv


## 12. Verificación final

Validación de integridad del pipeline: conteos por split, verificación de nulos en variables críticas, y comparación de medias de la variable objetivo entre splits para detectar distribuciones inconsistentes.

In [16]:
# ============================================================
# VERIFICACIÓN FINAL
# ============================================================

print("=" * 70)
print("VERIFICACIÓN FINAL DEL PIPELINE")
print("=" * 70)

splits = [("TRAIN", train_df), ("VALIDATION", val_df), ("TEST", test_df)]

for name, split_df in splits:
    cnt = split_df.count()
    print(f"\n{name}:")
    print(f"  Filas: {cnt:,}")

    # Rango de fechas
    dmin = split_df.select(min("Datetime")).collect()[0][0]
    dmax = split_df.select(max("Datetime")).collect()[0][0]
    print(f"  Fechas: {dmin} → {dmax}")

    # Verificar nulos en variable objetivo y lags críticos
    for c in ["Global_active_power", "Global_active_power_lag1h", "Voltage_lag1h"]:
        n = split_df.filter(col(c).isNull()).count()
        status = "✅" if n == 0 else "❌"
        print(f"  Nulos {c}: {n} {status}")

# Sanity check: medias de la variable objetivo
print("\n" + "=" * 70)
print("MEDIA DE GLOBAL_ACTIVE_POWER POR SPLIT")
print("=" * 70)
for name, split_df in splits:
    mu = split_df.select(avg("Global_active_power")).collect()[0][0]
    print(f"{name:12s}: {mu:.4f} kW")

# Verificar que las columnas escaladas existen
scaled_cols = [c for c in train_df.columns if "_scaled" in c]
print(f"\n📏 Columnas escaladas creadas: {len(scaled_cols)}")
print(f"   Ejemplos: {scaled_cols[:5]}")

print("\n" + "=" * 70)
print("PIPELINE DE PREPROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("=" * 70)
print("Entregables listos para notebooks 03-06:")
print("  • /content/preprocessed_data/train/")
print("  • /content/preprocessed_data/validation/")
print("  • /content/preprocessed_data/test/")
print("  • /content/preprocessed_data/min_max_scaler.json")

VERIFICACIÓN FINAL DEL PIPELINE

TRAIN:
  Filas: 17,743
  Fechas: 2006-12-23 17:00:00 → 2008-12-31 23:00:00
  Nulos Global_active_power: 0 ✅
  Nulos Global_active_power_lag1h: 0 ✅
  Nulos Voltage_lag1h: 0 ✅

VALIDATION:
  Filas: 8,760
  Fechas: 2009-01-01 00:00:00 → 2009-12-31 23:00:00
  Nulos Global_active_power: 0 ✅
  Nulos Global_active_power_lag1h: 0 ✅
  Nulos Voltage_lag1h: 0 ✅

TEST:
  Filas: 7,918
  Fechas: 2010-01-01 00:00:00 → 2010-11-26 21:00:00
  Nulos Global_active_power: 0 ✅
  Nulos Global_active_power_lag1h: 0 ✅
  Nulos Voltage_lag1h: 0 ✅

MEDIA DE GLOBAL_ACTIVE_POWER POR SPLIT
TRAIN       : 1.1028 kW
VALIDATION  : 1.0725 kW
TEST        : 1.0498 kW

📏 Columnas escaladas creadas: 61
   Ejemplos: ['Global_active_power_scaled', 'Global_reactive_power_scaled', 'Global_reactive_power_log_scaled', 'Voltage_scaled', 'Global_intensity_scaled']

PIPELINE DE PREPROCESAMIENTO COMPLETADO EXITOSAMENTE
Entregables listos para notebooks 03-06:
  • /content/preprocessed_data/train/
  • /

## Justificación del uso de PySpark en el preprocesamiento

El framework PySpark se utiliza para las operaciones donde realmente aporta valor en este pipeline:

| Operación | Herramienta | Justificación |
|---|---|---|
| Lectura CSV raw | pandas | Secuencial, C-optimizado, evita overhead JVM |
| Imputación temporal | pandas | `ffill`/`bfill` secuencial no es paralelizable |
| Agregación horaria | **PySpark** | `groupBy` + `agg` distribuye cálculo de 2M filas |
| Lags masivos | **PySpark** | `Window` + `lag` sobre dataset agregado eficiente |
| Ventanas móviles | **PySpark** | `Window.rowsBetween` para medias y std móviles |
| Codificación cíclica | **PySpark** | Vectorizado sobre columnas |
| Escalado global | **PySpark** | `min`/`max` paralelizados, aplicación vectorizada |
| División temporal | **PySpark** | `filter` + persistencia en Parquet |
| Persistencia | **PySpark** | Escritura Parquet nativa, particionada |

El pipeline demuestra dominio del flujo completo Big Data → Preprocesamiento → Ingeniería de características → Series de Tiempo, utilizando cada herramienta para lo que optimiza: pandas para operaciones secuenciales, PySpark para transformaciones vectorizadas y agregaciones distribuidas.

In [17]:
# ============================================================
# DESCARGA DEL DATASET PREPROCESADO
# ============================================================

import shutil

# Comprimir toda la carpeta
zip_path = "/content/preprocessed_data.zip"
shutil.make_archive("/content/preprocessed_data", 'zip', output_path)

print(f"📦 ZIP creado: {zip_path}")
print(f"   Tamaño: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")

# Descargar
files.download(zip_path)
print("✅ Descarga iniciada")

📦 ZIP creado: /content/preprocessed_data.zip
   Tamaño: 21.2 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada
